# 🔬 Pipeline de Detección Temprana de Melanoma v2.0
## Deep Learning Clínicamente Informado — Protocolo TRIPOD+AI

> **Interfaz visual sobre `run_pipeline.py` v2.0**  
> Arquitectura: DenseNet-121 | 3 condiciones principales (A/B/C) + 2 ablaciones (D/E)

---

### Decisiones de diseño clave (v2.0)
| Componente | v1.0 (incorrecto) | v2.0 (correcto) |
|---|---|---|
| Dataset | Sintético (PIL) | API ISIC real + sintético solo para CI |
| Split | Por imagen | Por paciente o agrupación mixta con advertencia |
| Umbral | Calculado en TEST (fuga) | Calibrado en VAL → frozen en TEST |
| Mitigación | Crop en inferencia | Masking en entrenamiento (COND-C) |
| Comparativa | Un modelo + crop | 3 modelos principales + ablaciones D/E |
| Test sesgo | Δ>0.1 arbitrario | Mann-Whitney U sobre activación periférica / AOS real si existe máscara |
| ΔAUC | Visual | DeLong test (p-value, IC 95%) |

### Experimentos incluidos
- **COND-A**: baseline sesgado, sin mitigación.
- **COND-B**: mitigación parcial con `RandomResizedCrop` en entrenamiento.
- **COND-C**: mitigación real con `MaskedISICDataset`.
- **ABLAC-D**: igual que COND-B, pero sin `WeightedSampler`.
- **ABLAC-E**: igual que COND-B, pero sin `Focal Loss`.

> El análisis clínico principal compara **A/B/C**. Las ejecuciones **D/E** se usan como ablaciones para aislar el aporte del sampler y de la pérdida focal.



---
## ⚙️ Setup


In [ ]:
# Instalación de dependencias
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "albumentations",
    "scikit-learn", "seaborn", "tqdm", "opencv-python", "requests"])
print("✅ Dependencias instaladas")


In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# Añadir src al path
sys.path.insert(0, os.path.abspath("./src"))
warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams["figure.dpi"] = 120

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")


---
## Configuración del Experimento v2.0

> Modifica `CONFIG` antes de ejecutar. Los valores por defecto usan la API ISIC real.


In [ ]:
CONFIG = {
    # Dataset
    "data_dir":           "data",
    "n_benign":           1500,
    "n_malignant":        500,
    "img_size":           224,
    # Entrenamiento
    "num_epochs":         20,
    "batch_size":         32,
    "lr_backbone":        1e-5,
    "lr_head":            1e-3,
    "weight_decay":       1e-4,
    "focal_gamma":        2.0,
    "patience":           5,
    "seed":               42,
    "freeze_strategy":    "partial",
    # Evaluación clínica
    "sensitivity_target": 0.90,
    "n_bootstrap":        1000,
    # Protección metodológica
    "require_strict_patient_grouping": True,
    # Rutas — un directorio por condición/ablación
    "checkpoint_dir_a":  "checkpoints/unmitigated",
    "checkpoint_dir_b":  "checkpoints/partial",
    "checkpoint_dir_c":  "checkpoints/masked",
    "checkpoint_dir_d":  "checkpoints/ablation_no_sampler",
    "checkpoint_dir_e":  "checkpoints/ablation_no_focal",
    "results_dir":       "results",
}

# Prevalencia real ISIC 2020 (~1.8%) — necesaria para calibrar PPV/NPV
REAL_PREVALENCE = 0.018
print(f"Configuración cargada. Sesión sobre: {DEVICE}")


---
## FASE 1: Dataset — Descarga Real y Split por Paciente


In [ ]:
from data_pipeline import (
    set_global_seed,
    ISICArchiveDownloader,
    ISICSyntheticUnitTest,
    patient_level_split,
    summarize_grouping_quality,
    validate_grouping_quality,
)

set_global_seed(CONFIG["seed"])

# DEMO_MODE: False → datos reales sin artefactos inyectados (sesgo nativo ISIC).
# True → artefactos sintéticos inyectados (proof-of-concept controlado, NO publicable).
DEMO_MODE = False

metadata_path = Path(CONFIG["data_dir"]) / "metadata.csv"

def _detect_real_data(df) -> bool:
    """Heurística robusta: ISIC real tiene columna isic_id o patient_id sin prefijo synth_."""
    if "isic_id" in df.columns:
        return True
    if "patient_id" in df.columns:
        return not df["patient_id"].astype(str).str.startswith("synth_").all()
    return False

if metadata_path.exists():
    print("📂 Cargando dataset existente...")
    df = pd.read_csv(metadata_path)
    using_real_data = _detect_real_data(df)
    if "split" not in df.columns and "patient_id" in df.columns:
        print("🧩 metadata.csv no incluye split. Recalculando agrupación...")
        df = patient_level_split(df, patient_col="patient_id")
        df.to_csv(metadata_path, index=False)
else:
    print("🌐 Intentando API ISIC...")
    downloader = ISICArchiveDownloader(output_dir=CONFIG["data_dir"])
    df = downloader.download_balanced_subset(
        n_benign=CONFIG["n_benign"],
        n_malignant=CONFIG["n_malignant"],
        img_size=CONFIG["img_size"],
        add_artifacts_for_demo=DEMO_MODE,
    )
    if df is None:
        print("⚠️  API ISIC no disponible → fallback sintético (solo CI/CD)")
        import warnings
        warnings.warn("Usando datos SINTÉTICOS. Los resultados NO son científicamente válidos.")
        gen = ISICSyntheticUnitTest()
        df = gen.generate(
            output_dir=CONFIG["data_dir"],
            n_benign=CONFIG["n_benign"],
            n_malignant=CONFIG["n_malignant"],
            img_size=CONFIG["img_size"],
        )
        using_real_data = False
    else:
        using_real_data = True

    # Split estratificado a NIVEL DE PACIENTE
    df = patient_level_split(df, patient_col="patient_id")
    df.to_csv(metadata_path, index=False)

grouping_summary = summarize_grouping_quality(df) if "patient_id" in df.columns else {}
if using_real_data and grouping_summary:
    print("\n🧾 Calidad del agrupamiento:")
    print(f"   Nivel declarado:     {grouping_summary['grouping_level']}")
    print(f"   Grupos únicos:       {grouping_summary['unique_groups']}")
    print(f"   Fallback por imagen: {grouping_summary['fallback_count']} ({grouping_summary['fallback_fraction']:.1%})")
    if CONFIG["require_strict_patient_grouping"]:
        validate_grouping_quality(grouping_summary, require_strict_patient_grouping=True)

dataset_summary = {
    "n_benign": int((df["label"] == 0).sum()),
    "n_malignant": int((df["label"] == 1).sum()),
    "study_prevalence": float(df["label"].mean()) if len(df) else 0.0,
}

print(f"\nDataset: {len(df)} imágenes")
print(f"Usando datos reales: {using_real_data}")
print(f"Ratio benigno:maligno = {dataset_summary['n_benign']}:{dataset_summary['n_malignant']}")
print(f"Prevalencia del estudio cargado: {dataset_summary['study_prevalence']:.1%}")
if not using_real_data:
    print("⚠️  Datos SINTÉTICOS cargados. Elimina data/metadata.csv para forzar descarga real.")
if DEMO_MODE:
    print("⚠️  DEMO_MODE=True: artefactos SINTÉTICOS inyectados. No publicar como sesgo real.")
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))


### 1.1 Visualización del dataset


In [ ]:
os.makedirs(CONFIG["results_dir"], exist_ok=True)

fig, axes = plt.subplots(2, 5, figsize=(20, 9))
fig.patch.set_facecolor("#0d1117")
fig.suptitle("Muestras del Dataset Dermatoscópico ISIC",
             color="white", fontsize=14, fontweight="bold")

clean_samples    = df[(df["has_artifact"] == False) & (df["split"] == "train")].head(5)
artifact_samples = df[(df["has_artifact"] == True)  & (df["split"] == "train")].head(5)

if len(artifact_samples) == 0:
    print("ℹ️  Sin imágenes con artefactos en train (DEMO_MODE=False). Mostrando solo limpias.")
    artifact_samples = clean_samples  # fallback visual

for row_idx, samples in enumerate([clean_samples, artifact_samples]):
    for col_idx, (_, row) in enumerate(samples.iterrows()):
        ax = axes[row_idx][col_idx]
        try:
            ax.imshow(Image.open(row["image_path"]).convert("RGB"))
        except Exception:
            ax.set_facecolor("#222")
        cls = "Maligno" if row["label"] == 1 else "Benigno"
        ax.set_title(f'{cls}\n{row.get("artifact_type","none")}', color="white", fontsize=8)
        ax.axis("off")

plt.tight_layout()
results_dir = CONFIG["results_dir"]
plt.savefig(f"{results_dir}/dataset_samples.png", dpi=100, bbox_inches="tight")
plt.show()


---
## FASE 2: Entrenamiento — 3 Condiciones Principales + 2 Ablaciones

> **Diseño experimental limpio (v2.0):**
> - **COND-A**: Baseline — modelo sesgado (sin ninguna mitigación)
> - **COND-B**: Mitigación parcial — `RandomResizedCrop` en entrenamiento
> - **COND-C**: Mitigación real — `MaskedISICDataset` (el modelo nunca ve píxeles de artefactos)
> - **ABLAC-D**: Misma ruta de datos que COND-B, pero sin `WeightedSampler`
> - **ABLAC-E**: Misma ruta de datos que COND-B, pero sin `Focal Loss`
>
> Cada ejecución entrena pesos **independientes** desde la misma inicialización ImageNet.
> La comparación principal para conclusiones clínicas sigue siendo **A/B/C**.



In [ ]:
from data_pipeline import create_dataloaders, get_transforms, FocalLoss, compute_focal_loss_alpha
from model import MelanomaClassifier, build_optimizer_and_scheduler, ModelTrainer

# 5-tupla: (data_cond, run_key, label, use_sampler, use_focal)
# run_key = clave única en models/all_metrics/aos_data para evitar colisiones.
CONDITIONS = [
    ("unmitigated", "unmitigated", "COND-A (Baseline sesgado)", True, True),
    ("partial", "partial", "COND-B (RandomResizedCrop)", True, True),
    ("masked", "masked", "COND-C (Masking real)", True, True),
    ("partial", "partial_no_sampler", "ABLAC-D (sin WeightedSampler)", False, True),
    ("partial", "partial_no_focal", "ABLAC-E (sin Focal Loss)", True, False),
]
PRIMARY_RUN_KEYS = ["unmitigated", "partial", "masked"]

models = {}
histories = {}
condition_meta = {}

for data_cond, run_key, cond_name, use_sampler, use_focal in CONDITIONS:
    print("\n" + "=" * 65)
    print(f"  Entrenando: {cond_name}")
    print("=" * 65)

    loaders, datasets = create_dataloaders(
        df,
        img_size=CONFIG["img_size"],
        batch_size=CONFIG["batch_size"],
        num_workers=0,
        condition=data_cond,
        use_weighted_sampler=use_sampler,
    )

    train_class_counts = datasets["train"].class_counts
    if use_focal:
        alpha = compute_focal_loss_alpha(train_class_counts)
        criterion = FocalLoss(alpha=alpha, gamma=CONFIG["focal_gamma"])
    else:
        import torch.nn as nn
        n_neg, n_pos = train_class_counts.get(0, 1), train_class_counts.get(1, 1)
        ce_weight = torch.tensor([1.0, n_neg / n_pos]).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=ce_weight)

    model = MelanomaClassifier(
        pretrained=True,
        freeze_layers=CONFIG["freeze_strategy"],
        dropout_rate=0.5,
    )
    optimizer, scheduler = build_optimizer_and_scheduler(
        model,
        lr_backbone=CONFIG["lr_backbone"],
        lr_head=CONFIG["lr_head"],
        weight_decay=CONFIG["weight_decay"],
        n_epochs=CONFIG["num_epochs"],
    )

    ckpt_dir = {
        "unmitigated": CONFIG["checkpoint_dir_a"],
        "partial": CONFIG["checkpoint_dir_b"],
        "masked": CONFIG["checkpoint_dir_c"],
        "partial_no_sampler": CONFIG["checkpoint_dir_d"],
        "partial_no_focal": CONFIG["checkpoint_dir_e"],
    }[run_key]

    trainer = ModelTrainer(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=DEVICE,
        checkpoint_dir=ckpt_dir,
        patience=CONFIG["patience"],
    )

    history = trainer.train(
        train_loader=loaders["train"],
        val_loader=loaders["val"],
        n_epochs=CONFIG["num_epochs"],
    )
    trainer.load_best_model()

    models[run_key] = (model, loaders, datasets)
    histories[run_key] = history
    condition_meta[run_key] = {
        "data_cond": data_cond,
        "label": cond_name,
        "use_sampler": use_sampler,
        "use_focal": use_focal,
    }

    print(f"  ✅ {cond_name} entrenado")


---
## FASE 3: XAI — Grad-CAM++ y Análisis de Sesgo Clever Hans


In [ ]:
from xai_gradcam import CleverHansAnalyzer
from data_pipeline import ISICDataset, get_transforms

model_a, loaders_a, datasets_a = models["unmitigated"]
analyzer_a = CleverHansAnalyzer(
    model=model_a, target_layer=model_a.get_features_layer(),
    device=DEVICE, use_gradcam_pp=True
)

test_df      = df[df["split"] == "test"].copy()
artifacts_df = test_df[test_df["has_artifact"] == True]
clean_df     = test_df[test_df["has_artifact"] == False]

transform_test = get_transforms("test", CONFIG["img_size"], condition="unmitigated")
xai_samples = []
for _, row in pd.concat([artifacts_df.head(3), clean_df.head(3)]).iterrows():
    img = Image.open(row["image_path"]).convert("RGB")
    img_tensor = transform_test(image=np.array(img))["image"]
    xai_samples.append((img_tensor, row["label"],
                        row.get("artifact_type", "none"), row["image_path"]))

fig_xai = analyzer_a.visualize_clever_hans_effect(
    xai_samples,
    save_path=f"{CONFIG['results_dir']}/gradcam_cond_a.png",
    figsize=(22, 5)
)
plt.show()


### 3.1 Análisis cuantitativo del sesgo (AOS real)

> `batch_bias_analysis` usa la **máscara binaria del artefacto** cuando está disponible
> para calcular el AOS real: fracción de activación Grad-CAM sobre píxeles del artefacto.
> Fallback a activación periférica (proxy geométrico) si no hay máscara.


In [ ]:
aos_data = {}
for data_cond, run_key, cond_name, _sampler, _focal in CONDITIONS:
    model_c, loaders_c, _ = models[run_key]
    analyzer_c = CleverHansAnalyzer(
        model=model_c, target_layer=model_c.get_features_layer(),
        device=DEVICE, use_gradcam_pp=True
    )
    test_path_ds = ISICDataset(
        df, split="test", img_size=CONFIG["img_size"],
        condition=data_cond, return_path=True
    )
    test_path_loader = torch.utils.data.DataLoader(
        test_path_ds, batch_size=8, shuffle=False, num_workers=0
    )
    summary = analyzer_c.batch_bias_analysis(
        loader=test_path_loader, df_metadata=df, max_samples=60
    )
    aos_data[run_key] = summary
    print(f"  {cond_name}: _mw_with n={len(summary.get('_mw_with',[]))},"
          f" _mw_without n={len(summary.get('_mw_without',[]))}")



---
## FASE 4: Evaluación Clínica con Umbral Frozen y Tests Estadísticos

> **Corrección crítica v2.0**: el umbral se calibra sobre VAL y se aplica
> **congelado** sobre TEST. Omitir `calibrate_threshold()` genera `RuntimeError`.


In [ ]:
import importlib, sys
import evaluation
importlib.reload(evaluation)
from evaluation import ClinicalEvaluator, StatisticalValidator
print("✅ Módulo evaluation recargado con éxito para aplicar los parches")

all_metrics = {}
all_cis     = {}

for data_cond, run_key, cond_name, _sampler, _focal in CONDITIONS:
    print(f"\n" + "-"*60)
    print(f"  Evaluando: {cond_name}")

    model_c, loaders_c, _ = models[run_key]
    evaluator = ClinicalEvaluator(
        model=model_c, device=DEVICE,
        clinical_sensitivity_target=CONFIG["sensitivity_target"]
    )

    # Umbral calibrado en VAL, NUNCA en TEST
    threshold = evaluator.calibrate_threshold(loaders_c["val"])
    print(f"  Umbral calibrado en VAL: {threshold:.4f}")

    metrics = evaluator.evaluate_test(loaders_c["test"])
    all_metrics[run_key] = metrics

    cis = evaluator.compute_bootstrap_cis(metrics, n_bootstrap=CONFIG["n_bootstrap"])
    all_cis[run_key] = cis

    auc_ci = all_cis[run_key].get("auc_roc", (0, 0, 0))
    print(f"  AUC-ROC: {metrics['AUC-ROC']:.4f} [{auc_ci[1]:.3f}–{auc_ci[2]:.3f}]")
    print(f"  Sensitivity: {metrics['Sensitivity']:.4f} | Specificity: {metrics['Specificity']:.4f}")



### 4.1 Tests estadísticos formales (DeLong + Mann-Whitney)


In [ ]:
# DeLong test: comparación de AUC entre las condiciones clínicas principales
print("\n[DeLong] COND-A vs. COND-C (masking real):")
delong_ac = StatisticalValidator.delong_test(
    y_true=all_metrics["unmitigated"]["_y_true"],
    y_prob_a=all_metrics["unmitigated"]["_y_prob"],
    y_prob_b=all_metrics["masked"]["_y_prob"],
)

print("\n[DeLong] COND-A vs. COND-B (crop parcial):")
delong_ab = StatisticalValidator.delong_test(
    y_true=all_metrics["unmitigated"]["_y_true"],
    y_prob_a=all_metrics["unmitigated"]["_y_prob"],
    y_prob_b=all_metrics["partial"]["_y_prob"],
)

# Mann-Whitney U sobre activación periférica normalizada.
# Si hay máscaras reales de artefacto, batch_bias_analysis también reporta AOS
# como descriptor adicional, pero no mezcla ambas métricas en el mismo test.
mw_results = {}
for _data_cond, run_key, cond_name, _sampler, _focal in CONDITIONS:
    print(f"\n[Mann-Whitney] {cond_name}: activación periférica con vs. sin artefacto")
    mw_with = aos_data[run_key].get("_mw_with", [])
    mw_without = aos_data[run_key].get("_mw_without", [])
    if len(mw_with) < 3 or len(mw_without) < 3:
        print(f"   ⚠️  n insuficiente: n_with={len(mw_with)}, n_without={len(mw_without)}")
        mw_results[run_key] = None
    else:
        mw = StatisticalValidator.mann_whitney_bias(mw_with, mw_without)
        mw_results[run_key] = mw


### 4.2 Tabla comparativa final con IC 95%


In [ ]:
cond_labels = {
    "unmitigated": "COND-A (sesgado)",
    "partial": "COND-B (crop)",
    "masked": "COND-C (masking)",
    "partial_no_sampler": "ABLAC-D (sin Sampler)",
    "partial_no_focal": "ABLAC-E (sin FocalLoss)",
}
key_metrics = ["AUC-ROC", "Sensitivity", "Specificity", "F1-Score", "FN"]
ordered_runs = ["unmitigated", "partial", "masked", "partial_no_sampler", "partial_no_focal"]

print("\n" + "=" * 120)
print("  COMPARACIÓN FINAL — 3 CONDICIONES PRINCIPALES + 2 ABLACIONES")
print("=" * 120)
print(f"{'Metrica':<20}", end="")
for cond in ordered_runs:
    print(f" {cond_labels[cond]:>22}", end="")
print()
print("-" * 120)

for m in key_metrics:
    print(f"{m:<20}", end="")
    for cond in ordered_runs:
        val = all_metrics[cond][m]
        ci_key = {
            "AUC-ROC": "auc_roc",
            "Sensitivity": "sensitivity",
            "Specificity": "specificity",
            "F1-Score": "f1",
        }.get(m)
        if ci_key and ci_key in all_cis[cond]:
            _, lo, hi = all_cis[cond][ci_key]
            print(f" {val:>7.4f} [{lo:.3f}–{hi:.3f}]", end="")
        else:
            print(f" {val:>22}", end="")
    print()

print("-" * 120)
print(f"  DeLong A vs C: ΔAUC = {delong_ac['delta_auc']:+.4f}, p = {delong_ac['p_value']:.4f} ({delong_ac['significance']})")
print(f"  DeLong A vs B: ΔAUC = {delong_ab['delta_auc']:+.4f}, p = {delong_ab['p_value']:.4f} ({delong_ab['significance']})")


### 4.3 Advertencia de prevalencia (TRIPOD+AI ítem 17)


In [ ]:
# PPV calibrado para prevalencia real (~1.8% ISIC 2020)
# Prevalencia REAL del dataset descargado (no del CONFIG hardcodeado)
n_mal_real = int(df["label"].sum())
n_total_real = len(df)
study_prevalence = n_mal_real / n_total_real if n_total_real > 0 else 0.0

print(f"\n⚠️  ADVERTENCIA DE PREVALENCIA")
print(f"   Prevalencia estudio:       {study_prevalence:.1%}")
print(f"   Prevalencia real ISIC2020: {REAL_PREVALENCE:.1%}")
print()

for data_cond, run_key, cond_name, _sampler, _focal in CONDITIONS:
    sens     = all_metrics[run_key]["Sensitivity"]
    spec     = all_metrics[run_key]["Specificity"]
    ppv_real = (sens * REAL_PREVALENCE) / (
        sens * REAL_PREVALENCE + (1 - spec) * (1 - REAL_PREVALENCE)
    )
    ppv_study = all_metrics[run_key].get("Precision", 0)
    print(f"  {cond_name}: PPV_calibrado={ppv_real:.4f}  (vs PPV_estudio={ppv_study:.4f})")



---
## Guardar Resultados


In [ ]:
def _safe(obj):
    if isinstance(obj, (int, float)):
        return obj
    try:
        import numpy as np
        if isinstance(obj, (np.integer, np.floating)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
    except ImportError:
        pass
    if isinstance(obj, list):
        return obj
    return str(obj)

results = {
    "config": CONFIG,
    "using_real_data": using_real_data,
    "scientific_use_allowed": grouping_summary.get("strict_patient_grouping", not using_real_data),
    "dataset_summary": dataset_summary,
    "grouping_quality": grouping_summary,
    "metrics": {
        k: {mk: float(mv) if hasattr(mv, "__float__") else mv for mk, mv in v.items() if not mk.startswith("_")}
        for k, v in all_metrics.items()
    },
    "delong_a_vs_c": delong_ac,
    "delong_a_vs_b": delong_ab,
    "mann_whitney": {k: v for k, v in mw_results.items() if v is not None},
    "prevalence_warning": {
        "study_prevalence": study_prevalence,
        "real_isic_prevalence": REAL_PREVALENCE,
    },
}
os.makedirs(CONFIG["results_dir"], exist_ok=True)
with open(f"{CONFIG['results_dir']}/final_results_v2.json", "w") as f:
    json.dump(results, f, indent=2, default=_safe)

print("✅ Resultados guardados en:", f"{CONFIG['results_dir']}/final_results_v2.json")
if not using_real_data:
    print("⚠️  RECORDATORIO: resultados sobre datos SINTÉTICOS, no publicables.")


---
## 📋 Discusión y Limitaciones (TRIPOD+AI)

### Hallazgos esperados
1. **COND-A (Baseline)**: AUC elevado solo en datos con artefactos → el modelo puede aprender sesgo Clever Hans.
2. **COND-B (Crop parcial)**: mitigación parcial del sesgo, pero no necesariamente lo elimina.
3. **COND-C (Masking real)**: sesgo menor y diagnóstico más centrado en la lesión.
4. **ABLAC-D**: cuantifica cuánto aporta `WeightedSampler` sobre la ruta parcial.
5. **ABLAC-E**: cuantifica cuánto aporta `Focal Loss` sobre la ruta parcial.

### Limitaciones reconocidas (TRIPOD+AI ítem 15)
| Limitación | Estado |
|---|---|
| Datos sintéticos (fallback) | ⚠️ Solo para CI/CD, no para publicación |
| Segmentación Otsu en COND-C | ⚠️ IoU vs ISIC 2018 Task 1 no medido |
| `patient_id` fallback a `isic_id` | ⚠️ No garantiza prevención completa de fuga |
| Multiple seeds no ejecutadas | ⚠️ Varianza de entrenamiento sin estimar |
| FocalLoss + WeightedSampler | ✅ Ablaciones D/E ya incluidas |
| n_malignant bajo en test | ⚠️ IC amplios, potencia estadística limitada |

